# **INTRODUCTION**

## **Import necessary library**

In [316]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

## **Access dataset**

In [317]:
df = pd.read_csv('../../data/raw/listing_data.csv', engine='python', on_bad_lines='warn')

/var/folders/dg/qmgw711x5_lds0sfww4_3my40000gn/T/ipykernel_64566/2389435519.py:1: ParserWarning: Skipping line 22: ',' expected after '"'

  df = pd.read_csv('../../data/raw/listing_data.csv', engine='python', on_bad_lines='warn')
/var/folders/dg/qmgw711x5_lds0sfww4_3my40000gn/T/ipykernel_64566/2389435519.py:1: ParserWarning: Skipping line 48: ',' expected after '"'

  df = pd.read_csv('../../data/raw/listing_data.csv', engine='python', on_bad_lines='warn')
/var/folders/dg/qmgw711x5_lds0sfww4_3my40000gn/T/ipykernel_64566/2389435519.py:1: ParserWarning: Skipping line 62: ',' expected after '"'

  df = pd.read_csv('../../data/raw/listing_data.csv', engine='python', on_bad_lines='warn')
/var/folders/dg/qmgw711x5_lds0sfww4_3my40000gn/T/ipykernel_64566/2389435519.py:1: ParserWarning: Skipping line 71: ',' expected after '"'

  df = pd.read_csv('../../data/raw/listing_data.csv', engine='python', on_bad_lines='warn')
/var/folders/dg/qmgw711x5_lds0sfww4_3my40000gn/T/ipykernel_64566/2389435519.

## **I. Features extraction**

In [318]:
cols_to_keep = [
    "listing_id",                               #
    "listing_name",
    "listing_type",
    "room_type",
    "superhost",
    "latitude",
    "longitude",
    "guests",
    "bedrooms",
    "beds",
    "baths",
    "registration",
    "amenities",
    "instant_book",
    "professional_management",
    "min_nights",
    "cancellation_policy",
    "checkin_time",
    "checkout_time",
    "guest_favorite",
    "exact_location",
    "cleaning_fee",
    "extra_guest_fee",
    "single_fee_structure",
    "num_reviews",
    "rating_overall",
    "rating_accuracy",
    "rating_checkin",
    "rating_cleanliness",
    "rating_communication",
    "rating_location",
    "rating_value",
    "ttm_revenue_native",
    "ttm_avg_rate_native",
    "ttm_occupancy",
    "ttm_adjusted_occupancy",
    "ttm_revpar_native",
    "ttm_adjusted_revpar_native",
    "ttm_avg_min_nights",
    "ttm_avg_length_of_stay",
    "ttm_reserved_days",
    "ttm_blocked_days",
    "ttm_available_days"
]

df_selected = df[cols_to_keep].copy()

**General check**

In [319]:
null_summary = (
    df_selected.isnull().sum()
    .reset_index()
)

null_summary.columns = ["column", "null_count"]

null_summary["null_percentage"] = (
    null_summary["null_count"] / len(df_selected) * 100
).round(2)

null_summary["dtype"] = null_summary["column"].map(df_selected.dtypes)

null_summary = null_summary[null_summary["null_count"] > 0]

null_summary = null_summary.sort_values(
    by="null_count",
    ascending=False
).reset_index(drop=True)

null_summary

,column,null_count,null_percentage,dtype
0,instant_book,224,84.21,object
1,checkin_time,84,31.58,object
2,checkout_time,82,30.83,object
3,bedrooms,75,28.20,float64
4,guest_favorite,73,27.44,object
5,exact_location,73,27.44,object
6,single_fee_structure,67,25.19,object
7,guests,66,24.81,float64
8,rating_accuracy,16,6.02,float64
9,rating_location,16,6.02,float64


## **II. Missing value handling**

In [320]:
cols_with_null = null_summary["column"].tolist()

unique_values_summary = pd.DataFrame({
    "column": cols_with_null,
    "unique_values": [
        df[col].dropna().unique().tolist()
        for col in cols_with_null
    ]
})

unique_values_summary

,column,unique_values
0,instant_book,"[True, False]"
1,checkin_time,"[2:00 PM, 10:00 AM, 9:00 AM, 3:00 PM, 1:00 PM,..."
2,checkout_time,"[11:00 AM, 12:00 PM, 10:00 AM, 2:00 PM, 1:00 PM]"
3,bedrooms,"[1.0, 2.0, 3.0, 4.0, 6.0, 5.0, 8.0, 7.0]"
4,guest_favorite,"[False, True]"
5,exact_location,"[False, True]"
6,single_fee_structure,"[True, False]"
7,guests,"[2.0, 5.0, 3.0, 4.0, 10.0, 6.0, 9.0, 1.0, 16.0..."
8,rating_accuracy,"[4.8, 5.0, 4.9, 4.7, 4.6, 4.0, 3.8, 4.4, 4.1, ..."
9,rating_location,"[4.9, 4.3, 4.8, 5.0, 4.2, 4.7, 4.6, 4.5, 4.0, ..."


**VI** <br>
<br>
- "instant_book": Đây là một trong những chức năng cài đặt của Airbnb. Nếu bật chức năng này, chúng ta sẽ có 2 lựa chọn Có/Không. Với "Có", khách có thể đặt phòng và thanh toán phí thủ công và chúng ta (chủ nhà) phải phục vụ họ dựa trên thời gian họ đặt phòng. Nếu chọn "Không" (tương tự như tắt), chúng ta có 24 giờ để chấp nhận yêu cầu từ Airbnb.<br>
**--------> Thay thế tất cả các giá trị null thành "False"**

- "check_in" và "check_out": Đây là khung thời gian cụ thể mà khách được phép vào bên trong chỗ ở. Nếu chỗ ở không ghi rõ thời gian nhận phòng và trả phòng, mặc định là 15:00 và trả phòng lúc 11:00 (giờ địa phương), trừ khi chủ nhà ghi chú khác.<br>
**--------> [check_in] Thay thế tất cả giá trị null thành "3:00PM"**<br>
**--------> [check_out] Thay thế tất cả giá trị null thành "11:00AM"**<br>

- "rating_accuracy", "rating_location", "rating_communication", "rating_cleanliness", "rating_checkin", "rating_value", "rating_overall", "num_reviews": Với các chỉ số đánh giá/hiệu suất, có thể khách hàng không để lại đánh giá và xếp hạng, hoặc họ đã để lại nhưng họ là đối tác của người đặt phòng.<br>
**--------> Thay thế tất cả giá trị null thành "0"**

- Các chỉ số/cột còn lại vẫn chưa rõ ràng. Chúng có thể liên quan đến nhau và có thể là những quan sát chất lượng kém (trong dữ liệu).<br>
**--------> Trước tiên hãy điều chỉnh tất cả các chỉ số/cột ở trên**

**EN** <br>
<br>
- "instant_book": This one is one of functions of Airbnb setting. If we turn it "ON", we will have 2 option Yes/No. With "Yes", guest can book anf pay the fee manually and we (host) have to serve them based on the time they booked. If we choose "No" (will be simmilar with turn it off), we have 24hr to accecpt the request from Airbnb.<br>
**--------> Replace all null value into "False"**

- "check_in" and "check_out": This are specific time frame that the guest is allowed to come in side the listing. If a listing doesn't specify check-in and checkout times, the defaults are 3:00 PM check-in and 11:00 AM checkout (local time), unless the host notes otherwise.<br>
**--------> [check_in] Replace all null value into "3:00 PM"**<br>
**--------> [check_out] Replace all null value into "11:00 AM"**<br>

- "rating_accuracy", "rating_location", "rating_communication", "rating_cleanliness", "rating_checkin", "rating_value", "rating_overall", "num_reviews": With rating/ performance metrics, it's possible that customers didn't leave reviews and rating, or that they did, but they were partner of booker <br>
**--------> Replace all null value into "0"**

- The remain metrics/ columns are still not clear. They can be realated together and can be the bad quality (in data) observasion.<br>
**--------> Adjust all the metrics/ columns above first** 

### **Turn 1:**

In [321]:
# Thay thế tất cả các giá trị null trong cột "instant_book" thành "False"
# Replace all the null values in columns "instant_book" to "False"
df_selected["instant_book"] = df_selected["instant_book"].fillna(False)

# Thay thế tất cả các giá trị null trong cột "checkin_time" thành "3:00 PM"
# Replace all the null values in columns "checkin_time" to "3:00 PM"
df_selected["checkin_time"] = df_selected["checkin_time"].fillna("3:00\u202fPM")

# Thay thế tất cả các giá trị null trong cột "checkout_time" thành "11:00 AM"
# Replace all the null values in columns "checkout_time" to "11:00 AM"
df_selected["checkout_time"] = df_selected["checkout_time"].fillna("11:00\u202fAM")

# Thay thế tất các các giá trị null trong các cột "rating_accuracy", "rating_location", "rating_communication", "rating_cleanliness", "rating_checkin", "rating_value", "rating_overall", "num_reviews" thành 0
# Replace all the null values in columns "rating_accuracy", "rating_location", "rating_communication", "rating_cleanliness", "rating_checkin", "rating_value", "rating_overall", "num_reviews" to 0
rating_cols = [
    "rating_accuracy",
    "rating_location",
    "rating_communication",
    "rating_cleanliness",
    "rating_checkin",
    "rating_value",
    "rating_overall",
    "num_reviews"
]

for col in rating_cols:
    df_selected[col] = df_selected[col].fillna(0)

# Xem xét lại các giá trị null sau khi đã thực hiện imputation (chỉ dành cho các cột còn lại có giá trị null)
# Review again the null values after imputation (only for the remaining columns that had null values)
remaining_null_summary = (
    df_selected[cols_with_null]
    .isnull()
    .sum()
)

remaining_null_summary = (
    remaining_null_summary[remaining_null_summary > 0]
    .reset_index()
)

remaining_null_summary.columns = ["column", "null_count"]

remaining_null_summary

/var/folders/dg/qmgw711x5_lds0sfww4_3my40000gn/T/ipykernel_64566/1428962275.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_selected["instant_book"] = df_selected["instant_book"].fillna(False)


,column,null_count
0,bedrooms,75
1,guest_favorite,73
2,exact_location,73
3,single_fee_structure,67
4,guests,66
5,extra_guest_fee,9
6,professional_management,7
7,beds,3


**VI** <br>
<br>
Hiện tại, trong những cột bị mất mát dữ liệu còn lại có 3 cột thể hiện thông tin về cơ sở vật chất ("bedrooms", "guests" và "beds") sẽ do chính người chủ cung cấp, nhưng sẽ có trường hợp họ không dùng tính năng cung cấp thông tin của nền tảng mà họ sẽ cung cấp thông tin trong phần mô tả của nơi ở. <br>
<br>
Đối với những cột liên quan đến thông tin dịch vụ như "single_fee_structure", "extra_guest_fee", nếu như chủ nhà không liệt kê chúng trong mô tả, thì chúng sẽ được trả về trế độ mắc định như ở trên Airbnb, cụ thể như sau:<br>
- "single_fee_structure": các giá trị thiếu sẽ được đưa về "True"<br>
- "extra_guest_fee": các giá trị thiếu sẽ đưọc đưa về "False"<br>

Đối với những cột liên quan đến danh hiệu hay thông tin khác do chính nền tàng cung cấp sẽ được đưa về chế độ mắc định, cụ thể:
- "guest_favorite": các giá trị thiếu sẽ được đưa về "False"<br>
- "exact_location": các giá trị thiếu sẽ đưọc đưa về "False"<br>

Đặc biệt, đối với "professional_management", đây là thông tin thể hiện các listing được quản lý bởi loại host nào (business hay là cá nhân). Thông tin này cũng là không được thể hiện được đối với khách hàng, nhưng nó có thể thể hiên qua tên của các host. Cột này sẽ được xem xét thêm với tên của các host để cân nhắc.

**EN** <br>
<br>
Currently, among the remaining columns with lost data, there are 3 columns showing information about the listing("bedrooms", "guests", and "beds") which would be provided by the host, but there may be cases where they don't use the platform's information provision feature and instead provide the information in the property description. <br>
<br>
For columns related to service information such as "single_fee_structure", "extra_guest_fee", if the host does not list them in the description, they will be returned to the default mode as on Airbnb, specifically as follows:<br>
- "single_fee_structure": missing values ​​will be returned to "True"<br>
- "extra_guest_fee": missing values ​​will be returned to "False"<br>

For columns related to titles or other information provided by the platform itself, they will be returned to the default mode, specifically:
- "guest_favorite": missing values ​​will be returned to "False"<br>
- "exact_location": missing values ​​will be returned to "False"<br>

Specifically, for "professional_management," this information indicates which type of host manages the listings (business or individual). This information is also not visible to customers, but it can be shown through the names of the hosts. This column will be further considered in relation to the names of the hosts.

### **Turn 2:**

In [322]:
# Tạo bảng tạm thời chỉ chứa cột "host_name", "description", "listing_type", "room_type", "bedrooms", "guest", "beds", và "professional_management", chỉ chứa các giá trị null để có thể xem xét phương án xử lý mất dữ liệu.
# Create a temporary table containing only the columns "host_name", "description", "listing_type", "room_type", "bedrooms", "guest", "beds", and "professional_management", with only null values, so that we can consider how to handle data loss.
cols_check_null = [
    "listing_id",
    "listing_type",
    "room_type",
    "bedrooms",
    "guests", 
    "beds",
    "professional_management"
]

null_tables = {}

for col in cols_check_null:
    null_tables[col] = df_selected[
        df_selected[col].isnull()
    ][cols_check_null]

#### **Bedrooms**

In [323]:
# Những dòng bị thiếu dữ liệu ở cột "bedrooms"
# The row is getting missing value at "bedrooms" column
df_br = null_tables["bedrooms"]
df_br

,listing_id,listing_type,room_type,bedrooms,guests,beds,professional_management
1,1271743,Private room in home,private_room,NaN,NaN,1.0,False
2,3354986,Private room in home,private_room,NaN,NaN,2.0,False
5,6208123,Entire serviced apartment,entire_home,NaN,3.0,1.0,False
6,6531350,Private room in home,private_room,NaN,NaN,1.0,False
11,10540525,Private room in home,private_room,NaN,NaN,1.0,False
...,...,...,...,...,...,...,...
230,32269958,Private room in townhouse,private_room,NaN,NaN,1.0,False
231,32439734,Entire rental unit,entire_home,NaN,2.0,1.0,False
237,32799257,Private room in rental unit,private_room,NaN,NaN,1.0,False
252,33783587,Private room in rental unit,private_room,NaN,NaN,1.0,False


In [324]:
df_br["listing_type"].unique()

array(['Private room in home', 'Entire serviced apartment',
       'Private room in rental unit', 'Private room in villa',
       'Shared room in hostel', 'Private room in condo',
       'Entire rental unit', 'Private room in townhouse', 'Entire home',
       'Entire townhouse', 'Private room in serviced apartment'],
      dtype=object)

**VI** <br>
<br>
Ta thấy có những nhóm listing là "room", điều này thể hiện bedrooms sẽ là listing cho nên chúng ta có thể ghi nhận chúng là giá trị 1. <br>
<br>
Ngoài ra sẽ còn một số lising khác dạng với "room" sẽ cần cân nhắc thêm ở phần "description". Cụ thể hơn, thông tin về cột "bedrooms" sẽ được ấy từ chữ số trước chữ "bedrooms" ở phần "description".

**EN** <br>
<br>
We see that there are listing groups of "room", which indicates that bedrooms will be listings, so we can record them as a value of 1. <br>
<br>
In addition, there will be some other listings of the same type as "room" that will need further consideration in the "description" section. More specifically, the information for the "bedrooms" column will be taken from the digit before the word "bedrooms" in the "description" section.

In [325]:
df_de = df[["listing_id", "description"]]

df_br1 = pd.merge(df_br, df_de, on="listing_id", how="left")

extracted_bedrooms = (
    df_br1["description"]
    .astype("string")
    .str.extract(r"(?i)(\d+)\s*[- ]?\s*bedrooms?\b")[0]
    .astype("float")
)

# Chỉ fill những dòng bedrooms đang null và có tìm được số từ description
mask = df_br1["bedrooms"].isnull() & extracted_bedrooms.notnull()

df_br1.loc[mask, "bedrooms"] = extracted_bedrooms.loc[mask]

print("Rows to fill:", mask.sum())

Rows to fill: 1


In [326]:
# Fill bedrooms = 1 where listing_type contains "room" and bedrooms is null

mask_room_listing = (
    df_br1["bedrooms"].isnull() &
    df_br1["listing_type"].astype("string").str.contains("room", case=False, na=False)
)

print("Rows to fill:", mask_room_listing.sum())

df_br1.loc[mask_room_listing, "bedrooms"] = 1

Rows to fill: 59


In [327]:
df_br1[
    df_br1["bedrooms"].isnull()
]

,listing_id,listing_type,room_type,bedrooms,guests,beds,professional_management,description
2,6208123,Entire serviced apartment,entire_home,NaN,3.0,1.0,False,Welcome to enjoy this peaceful and centrally-l...
7,13790280,Entire serviced apartment,entire_home,NaN,2.0,1.0,False,"If you ask any Hanoi-holic, you will heard the..."
14,19682586,Entire rental unit,entire_home,NaN,2.0,1.0,False,A spacious studio located in a very secured an...
38,25507211,Entire serviced apartment,entire_home,NaN,2.0,1.0,False,The building is designed in a temporary archit...
43,26013867,Entire home,entire_home,NaN,4.0,3.0,False,NaN
52,28823009,Entire rental unit,entire_home,NaN,3.0,1.0,False,NaN
53,29151554,Entire townhouse,entire_home,NaN,4.0,3.0,False,"Located in Hàng Thiếc Street, the noisiest str..."
56,29689988,Entire serviced apartment,entire_home,NaN,2.0,1.0,False,"+ If you ask any Hanoi-holic, you will heard t..."
57,29948276,Entire serviced apartment,entire_home,NaN,2.0,1.0,False,"If you ask any Hanoi-holic, you will heard the..."
58,30021069,Entire rental unit,entire_home,NaN,NaN,1.0,False,*We are Rosemary Homestay. We hope to welcome...


**VI** <br>
<br>
Do không có thêm thông tin tương ứng nào để suy luận chính xác số lượng phòng ngủ mà mỗi listing có. Thông tin về số lượng giường sẽ được dựa vào để giải quyết giá thị thiếu về phòng ngủ 1 cách tương đối, 1 giường ngủ sẽ tương đương với 1 phòng ngủ.

**EN**<br>
<br>
Since there is no further corresponding information to accurately infer the number of bedrooms each listing has, the information regarding the number of beds will be used to determine the market price for bedrooms relatively, with 1 bed equivalent to 1 bedroom.

In [328]:
mask_fill_bedrooms_beds = (
    df_br1["bedrooms"].isnull() &
    df_br1["beds"].notnull()
)

print("Rows to fill:", mask_fill_bedrooms_beds.sum())

df_br1.loc[mask_fill_bedrooms_beds, "bedrooms"] = df_br1.loc[
    mask_fill_bedrooms_beds,
    "beds"
]

Rows to fill: 15


In [329]:
df_br1 = df_br1[["listing_id","bedrooms"]]

**VI**<br>
<br>
Dữ liệu thiếu ở cột "bedrooms" đã được giải quyết hoàn toàn và sẽ được bổ sung vào khung dữ liệu chính "df_selected"

**EN**<br>
<br>
The missing data in the "bedrooms" column has been completely resolved and will be added to the main data frame "df_selected"

In [330]:
bedrooms_map = df_br1.set_index("listing_id")["bedrooms"]


df_selected["bedrooms"] = df_selected["bedrooms"].fillna(
    df_selected["listing_id"].map(bedrooms_map)
)

#### **Beds**

In [331]:
df_bd = null_tables["beds"]
df_bd

,listing_id,listing_type,room_type,bedrooms,guests,beds,professional_management
83,21711477,Private room in tiny home,private_room,1.0,NaN,NaN,False
169,29191158,Train,entire_home,1.0,2.0,NaN,False
213,31303484,Private room in home,private_room,1.0,NaN,NaN,False


**VI** <br>
<br>
Trong phần mô tả của các listing thiếu thông tin về số giường đều cung cấp số thông tin là 1 giuờng <br>
**-------> Thay thế các giá trị null thành "1"**

**EN** <br>
<br>
In the descriptions of listings lacking information about the number of beds, the number provided is 1 bed.

**-------> Replace null values ​​with "1"**

In [332]:
df_selected["beds"] = df_selected["beds"].fillna(1)

#### **Guests**

In [333]:
df_gt = null_tables["guests"]
df_gt

,listing_id,listing_type,room_type,bedrooms,guests,beds,professional_management
1,1271743,Private room in home,private_room,NaN,NaN,1.0,False
2,3354986,Private room in home,private_room,NaN,NaN,2.0,False
6,6531350,Private room in home,private_room,NaN,NaN,1.0,False
11,10540525,Private room in home,private_room,NaN,NaN,1.0,False
14,11350161,Private room in rental unit,private_room,NaN,NaN,2.0,False
...,...,...,...,...,...,...,...
237,32799257,Private room in rental unit,private_room,NaN,NaN,1.0,False
241,33122984,Private room in rental unit,private_room,2.0,NaN,2.0,False
252,33783587,Private room in rental unit,private_room,NaN,NaN,1.0,False
254,33849273,Private room in townhouse,private_room,3.0,NaN,3.0,False


**VI** <br>
<br>
Không có thông tin liên quan được cung cấp về số lượng khách tôí đa mà các listing đang bị thiếu dữ liệu ở cột "guest". Thông tin này sẽ được giả lập theo tiêu chuẩn 2 người 1 giường và số lượng khách tối đa mà listing có thể nhận là "beds" nhân với 2.

**EN**

<br>
No relevant information is provided regarding the maximum number of guests that listings with missing data in the "guest" column can accommodate. This information will be assumed based on a standard of 2 people per bed, and the maximum number of guests a listing can accept is "beds" multiplied by 2.

In [334]:
mask_fill_guests_by_beds = (
    df_selected["guests"].isnull() &
    df_selected["beds"].notnull()
)

print("Rows to fill:", mask_fill_guests_by_beds.sum())

df_selected.loc[mask_fill_guests_by_beds, "guests"] = (
    df_selected.loc[mask_fill_guests_by_beds, "beds"] * 2
)

Rows to fill: 66


#### **Professional Management**

In [335]:
df_pm = null_tables["professional_management"]

df_de = df[["listing_id", "host_name"]]

df_pm1 = pd.merge(df_pm, df_de, on="listing_id", how="left")

df_pm1


,listing_id,listing_type,room_type,bedrooms,guests,beds,professional_management,host_name
0,21853744,Private room in home,private_room,NaN,NaN,1.0,NaN,Thanh
1,27475518,Private room in home,private_room,NaN,NaN,1.0,NaN,Hue
2,29310640,Entire rental unit,entire_home,1.0,4.0,2.0,NaN,Inam
3,31078539,Entire rental unit,entire_home,1.0,2.0,1.0,NaN,Jenny
4,31866373,Private room in serviced apartment,private_room,1.0,2.0,1.0,NaN,David Nam
5,32553010,Entire rental unit,entire_home,1.0,2.0,1.0,NaN,Thắng
6,33266882,Private room in serviced apartment,private_room,1.0,2.0,1.0,NaN,David Nam


**VI**<br>
<br>
Toàn bộ đều là tên cá nhân làm host, cho nên những listing này sẽ được xếp vào là "False"<br>
**-------> Replace the null value with "False"**

**EN**<br>
<br>
All of these are personal names used as hosts, so these listings will be classified as "False"<br>
**-------> Replace the null value with "False"**

In [336]:
df_selected["professional_management"] = df_selected["professional_management"].fillna(False)

/var/folders/dg/qmgw711x5_lds0sfww4_3my40000gn/T/ipykernel_64566/3604572936.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_selected["professional_management"] = df_selected["professional_management"].fillna(False)


#### **The rest**

In [337]:
df_selected["single_fee_structure"] = df_selected["single_fee_structure"].fillna(True)
df_selected["extra_guest_fee"] = df_selected["extra_guest_fee"].fillna(False)
df_selected["guest_favorite"] = df_selected["guest_favorite"].fillna(False)
df_selected["exact_location"] = df_selected["exact_location"].fillna(False)

/var/folders/dg/qmgw711x5_lds0sfww4_3my40000gn/T/ipykernel_64566/4007560843.py:1: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_selected["single_fee_structure"] = df_selected["single_fee_structure"].fillna(True)
/var/folders/dg/qmgw711x5_lds0sfww4_3my40000gn/T/ipykernel_64566/4007560843.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_selected["guest_favorite"] = df_selected["guest_favorite"].fillna(False)
/var/folders/dg/qmgw711x5_lds0sfww4_3my40000gn/T/ipykernel_64566/4007560843.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, 

### **III. Feature Engineering**

In [338]:
dtype_unique_summary = pd.DataFrame({
    "column": df_selected.columns,
    "dtype": df_selected.dtypes.astype(str).values,
    "n_unique": [df_selected[col].nunique(dropna=False) for col in df_selected.columns],
    "unique_values": [
        df_selected[col].dropna().unique().tolist()
        for col in df_selected.columns
    ]
})

dtype_unique_summary

,column,dtype,n_unique,unique_values
0,listing_id,int64,266,"[506824, 1271743, 3354986, 4612134, 5842921, 6..."
1,listing_name,object,263,"[Modern Apartment in Central Hanoi, Peaceful H..."
2,listing_type,object,22,"[Entire rental unit, Private room in home, Ent..."
3,room_type,object,4,"[entire_home, private_room, hotel_room, shared..."
4,superhost,bool,2,"[True, False]"
5,latitude,float64,175,"[21.0303, 21.0511, 21.03, 21.0341, 21.019, 21...."
6,longitude,float64,185,"[105.8504, 105.8878, 105.7859, 105.8468, 105.8..."
7,guests,float64,13,"[2.0, 4.0, 5.0, 3.0, 10.0, 6.0, 9.0, 1.0, 16.0..."
8,bedrooms,float64,8,"[1.0, 2.0, 3.0, 4.0, 6.0, 5.0, 8.0, 7.0]"
9,beds,float64,11,"[1.0, 2.0, 3.0, 5.0, 6.0, 7.0, 4.0, 11.0, 8.0,..."


**VI**<br>
<br>
Các đặc điểm sẽ được gần như giữ nguyên do đã phù hợp với cho mục đích của phần phân tích. Tuy nhiên, cột "amenities" vẫn cần được biến đối để làm rõ thông tin về các cơ sở vật chất của listing khi phân tích.<br>
<br>
Dựa vào cột "amenities", số lượng dịch vụ mà listing đó cung cấp sẽ được đếm.

**EN**<br>
<br>
The characteristics will be kept almost unchanged as they are already suitable for the purposes of this analysis. However, the "amenities" column still needs to be modified to clarify information about the listing's facilities during the analysis.<br>
<br>
Based on the "amenities" column, the number of services that the listing provides will be counted.

In [339]:
df_selected["amenities_count"] = (
    df_selected["amenities"]
    .fillna("")
    .astype(str)
    .apply(lambda x: 0 if x.strip() == "" else x.count(",") + 1)
)

In [340]:
import numpy as np
import pandas as pd

# Tọa độ trung tâm Hà Nội
hanoi_center_lat = 21.02808
hanoi_center_lon = 105.85404

# Bán kính Trái Đất
R = 6371

# Ép kiểu dữ liệu
df_selected["latitude"] = pd.to_numeric(df_selected["latitude"], errors="coerce")
df_selected["longitude"] = pd.to_numeric(df_selected["longitude"], errors="coerce")

# Đổi sang radian
lat1 = np.radians(hanoi_center_lat)
lon1 = np.radians(hanoi_center_lon)

lat2 = np.radians(df_selected["latitude"])
lon2 = np.radians(df_selected["longitude"])

# Haversine distance
dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    np.sin(dlat / 2) ** 2
    + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
)

c = 2 * np.arcsin(np.sqrt(a))

# Tạo cột khoảng cách dạng số hàng đơn vị / hàng chục
df_selected["center_mark"] = (R * c).round(2)

### **IV. Exporting dataset**

In [342]:
df_selected.to_csv('../../data/processed/listing_data.csv', index = False)